# transforEmotion in Google Colab

[transforEmotion](https://github.com/atomashevic/transforEmotion) scores emotions in text, images and video with transformer models, from R. The models are zero-shot: you choose the emotion labels, and no training data is needed. Everything runs inside this runtime; no data is sent to an external service.

**Runtime.** The notebook runs on the free CPU runtime. To use a GPU, choose **Runtime > Change runtime type > T4 GPU** before running the first cell; the package detects it and uses it.

**Time.** Run the cells in order. The first run downloads the R packages, Python and the models, which takes about 5–8 minutes in total. After that, each analysis takes seconds.

## 1. Install the package

Colab builds R packages from source by default, which is slow. The first two lines switch to precompiled packages for Colab's Ubuntu release, from [Posit Package Manager](https://p3m.dev).

In [ ]:
codename <- sub("VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE))
options(repos = c(CRAN = sprintf("https://p3m.dev/cran/__linux__/%s/latest", codename)), Ncpus = 2)

install.packages("remotes")
remotes::install_github("atomashevic/transforEmotion", upgrade = "never")

## 2. Load the package and set up Python

You do not install Python yourself. `setup_modules()` does it the first time:

- downloads [uv](https://docs.astral.sh/uv/), a fast Python package manager;
- installs Python 3.12, PyTorch, transformers and the other Python packages (about 500 MB, or 3 GB on a GPU runtime);
- downloads the default text, image and sentence-similarity models (about 1 GB).

This takes 1–3 minutes. In the same runtime it is instant afterwards, also after **Runtime > Restart session**. Deleting the runtime deletes the downloads, and a new runtime repeats this step.

Calling `setup_modules()` is optional: each function sets up what it needs on first use. Running it here keeps the waiting in one place.

In [ ]:
library(transforEmotion)
setup_modules()

## 3. Emotions in text

`transformer_scores()` scores each text against the classes you give it. For each text, the scores are probabilities across the classes and sum to 1.

The example scores five items from the friendliness facet of the NEO-PI-R personality inventory against the six facets of extraversion. The default model is [DistilRoBERTa](https://huggingface.co/cross-encoder/nli-distilroberta-base); `transformer = "facebook-bart"` uses a larger, slower model.

In [ ]:
data(neo_ipip_extraversion)
text <- neo_ipip_extraversion$friendliness[1:5]
classes <- c("friendly", "gregarious", "assertive", "active", "excitement", "cheerful")

scores <- transformer_scores(text = text, classes = classes)
round(do.call(rbind, scores), 3)

Any text and any labels work the same way. For short everyday texts and basic emotions, the larger [BART](https://huggingface.co/facebook/bart-large-mnli) model is clearly more accurate than the default; its first use downloads it (1.6 GB, about a minute).

In [ ]:
scores <- transformer_scores(
  text = c(
    "We won the final! This is the best day of my life.",
    "I miss my grandmother every day.",
    "Stop lying to me, I am sick of it!",
    "I heard a noise downstairs and I am shaking."
  ),
  classes = c("joy", "sadness", "anger", "fear", "surprise"),
  transformer = "facebook-bart"
)
round(do.call(rbind, scores), 3)

## 4. Similarity between texts

`sentence_similarity()` compares texts by meaning. Values are cosine similarities: close to 1 for texts with the same meaning, close to 0 for unrelated texts.

In [ ]:
sentence_similarity(
  text = "The cat sat on the mat",
  comparison_text = c("A cat is sitting on a rug", "Stock markets fell sharply today")
)

## 5. Facial expressions in images

`image_scores()` finds the largest face in the image and scores its expression against the labels, using OpenAI's [CLIP](https://huggingface.co/openai/clip-vit-base-patch32) model. It takes a file path or a URL.

To analyse your own image, upload it in the **Files** panel (folder icon on the left) and use its path, for example `"/content/photo.jpg"`.

In [ ]:
image <- system.file("extdata", "boris-1.png", package = "transforEmotion")
IRdisplay::display_png(file = image)

emotions <- c("happiness", "sadness", "anger", "fear", "surprise", "neutral")
round(image_scores(image, classes = emotions), 3)

## 6. Facial expressions in video

`video_scores()` takes `nframes` frames from the video, one every `ffreq` frames (15 by default), and scores the face in each, giving one row per frame. It takes a video file or a YouTube URL. YouTube often blocks downloads from Colab's servers, so for your own videos, upload the file in the **Files** panel and use its path.

For this demo, the next cell makes an 8-second clip from the package's two example images.

In [ ]:
cv2 <- reticulate::import("cv2", convert = FALSE)
frame1 <- cv2$imread(system.file("extdata", "boris-1.png", package = "transforEmotion"))
dims <- reticulate::py_to_r(frame1$shape)
size <- reticulate::tuple(dims[[2]], dims[[1]])
frame2 <- cv2$resize(cv2$imread(system.file("extdata", "boris-2.png", package = "transforEmotion")), size)

video <- "/content/demo.mp4"
writer <- cv2$VideoWriter(video, cv2$VideoWriter_fourcc("m", "p", "4", "v"), 25L, size)
for (i in 1:200) writer$write(if (i <= 100) frame1 else frame2)
invisible(writer$release())

In [ ]:
result <- video_scores(video, classes = emotions, nframes = 10, save_dir = tempdir())
round(result, 3)

matplot(as.matrix(result), type = "l", lty = 1, lwd = 2, xlab = "Frame", ylab = "Score")
legend("topright", legend = names(result), col = seq_along(result), lty = 1, lwd = 2, cex = 0.8)

## 7. Questions about documents (optional)

`rag()` answers a question about your texts, or PDFs in a folder (`path = `), with a small language model that runs in this runtime. The first call installs the LlamaIndex packages and downloads TinyLLAMA (about 2 GB). After that, this example takes about a minute on the CPU runtime and a few seconds on a T4 GPU.

In [ ]:
documents <- c(
  "The team won the championship and fans celebrated in the streets.",
  "After the loss, supporters left the stadium in silence and tears."
)
rag(text = documents, query = "What emotions are described?", transformer = "TinyLLAMA")

## Next steps

- **Your data.** Upload files in the **Files** panel. They are deleted with the runtime, so save results, for example with `write.csv(result, "/content/results.csv")`, and download them from the Files panel.
- **Other models.** `transformer_scores(transformer = "facebook-bart")`, `image_scores(model = "oai-large")`, and `list_vision_models()` for the available image models.
- **More functions.** `vad_scores()` for valence, arousal and dominance, and `evaluate_emotions()` for evaluating classifications. See the [README](https://github.com/atomashevic/transforEmotion) and `help(package = "transforEmotion")`.
- **Citation.** `citation("transforEmotion")`